# 指数权重清单推算

使用最近一次官方权重清单、成分股日收盘价和公司送转股行为，推算目标交易日的指数权重清单。

## 参数设置

In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import safe_query as gogoal_query
from utils.weight_projection import (
    fetch_corporate_actions,
    fetch_stock_closes_range,
    load_or_download_weight_source,
    make_projection_output_dir,
    normalize_date_key,
    project_weights_by_close_and_actions,
    save_projection_outputs,
)

# 指数代码，例如 000016 / 000300 / 000905 / 000852。
INDEX_CODE = "000300"

# file_path 为空时，下载中证官网当前最新样本权重文件；不为空时读取该本地文件。
# 支持已标准化 CSV（含 stock_code/raw_weight_pct/closeweight_data_date）或中证 Excel。
file_path = './data/for_test/index_weights.csv'

# 目标日期，YYYYMMDD。
target_date = "20260630"

# 输出目录；None 时自动写入 ./data/weights_projection/。
# 每个交易日保存为 f"{INDEX_CODE}-{current_date}.csv"。
output_dir = None


In [2]:
from xtquant import xtdata
xtdata.reconnect(port = 58610)

***** xtdata连接成功 2026-07-03 13:19:15*****
服务信息: {'tag': 'qmt_research', 'version': '1.0'}
服务地址: 127.0.0.1:58610
数据路径: E:\迅投极速交易终端睿智融科版\datadir
设置xtdata.enable_hello = False可隐藏此消息



## 读取或下载源权重清单

In [3]:
download_dir = PROJECT_ROOT / "data" / INDEX_CODE / "weight_projection" / "source_files"
source_weights, source_weight_path = load_or_download_weight_source(
    index_code=INDEX_CODE,
    file_path=file_path or None,
    output_dir=download_dir,
)
source_weight_date = normalize_date_key(source_weights["closeweight_data_date"].iloc[0])
target_date = normalize_date_key(target_date)

print("Source weight path:", source_weight_path)
print("Source weight date:", source_weight_date)
print("Target date:", target_date)
print("Source weight rows:", len(source_weights))
print("Source weight sum pct:", source_weights["raw_weight_pct"].sum())
display(source_weights.head())


Source weight path: data\for_test\index_weights.csv
Source weight date: 20260615
Target date: 20260630
Source weight rows: 300
Source weight sum pct: 99.998


,stock_code,stock_name,raw_weight_pct,closeweight_data_date,source_path
0,000001.SZ,平安银行,0.386,20260615,data\for_test\index_weights.csv
1,000002.SZ,万科A,0.077,20260615,data\for_test\index_weights.csv
2,000063.SZ,中兴通讯,0.432,20260615,data\for_test\index_weights.csv
3,000100.SZ,TCL科技,0.346,20260615,data\for_test\index_weights.csv
4,000157.SZ,中联重科,0.136,20260615,data\for_test\index_weights.csv


## 查询日收盘价与公司行为

In [4]:
#Task5
def normalize_stock_code(value) -> str:
    raw = str(value).strip().upper()
    if raw.endswith((".SH", ".SZ", ".BJ")):
        return raw
    digits = raw.split(".")[0].zfill(6)
    return f"{digits}.SH" if digits.startswith(("5", "6", "9")) else f"{digits}.SZ"

def strip_market_suffix(value) -> str:
    return str(value).strip().upper().split(".")[0].zfill(6)

def normalize_trade_date_dash(value) -> str:
    return pd.Timestamp(value).strftime("%Y-%m-%d")

def sql_literal(value) -> str:
    return "'" + str(value).replace("'", "''") + "'"


def sql_in(values) -> str:
    vals = list(values)
    if not vals:
        raise ValueError("SQL IN 列表为空")
    return ", ".join(sql_literal(v) for v in vals)

def fetch_gogoal_corporate_actions(stock_codes: list[str], start_date: str, end_date: str) -> pd.DataFrame:
    stripped_codes = [strip_market_suffix(code) for code in stock_codes]
    sql = f"""
        SELECT
            stock_code,
            stock_name,
            declare_date,
            xr_xd_date AS ex_date,
            beftax_maxcashdiv,
            beftax_mincashdiv,
            aftax_cashdiv,
            stockdiv_ratio,
            trans_ratio,
            bonus_ratio,
            is_newest,
            is_valid
        FROM bas_stk_hisdistribution
        WHERE stock_code IN ({sql_in(stripped_codes)})
          AND xr_xd_date >= {sql_literal(normalize_trade_date_dash(start_date))}
          AND xr_xd_date <= {sql_literal(normalize_trade_date_dash(end_date))}
          AND is_valid = 1
    """
    raw = gogoal_query(sql, output_format="dataframe")
    if raw.empty:
        return pd.DataFrame(columns=["stock_code", "ex_date"])
    raw["stock_code"] = raw["stock_code"].map(normalize_stock_code)
    raw["ex_date"] = pd.to_datetime(raw["ex_date"], errors="coerce").dt.strftime("%Y-%m-%d")
    return raw

stock_codes = source_weights["stock_code"].tolist()
# future_date = (pd.Timestamp(last_fitting_date) + timedelta(days=1)).strftime("%Y%m%d")
started_at = datetime.now().astimezone()
import_time = started_at.strftime("%Y%m%d-%H%M")

df_actions_gogoal = fetch_gogoal_corporate_actions(stock_codes, source_weight_date, target_date)
df_actions_gogoal["import_time"] = import_time

xt_action_frames = []
for stock_code in stock_codes:
    frame = xtdata.get_divid_factors(stock_code, source_weight_date, target_date)
    if frame is not None and not frame.empty:
        item = frame.copy().reset_index().rename(columns={"index": "ex_date"})
        item.insert(0, "stock_code", stock_code)
        xt_action_frames.append(item)
df_actions_xt = pd.concat(xt_action_frames, ignore_index=True) if xt_action_frames else pd.DataFrame(columns=["stock_code", "ex_date"])
if not df_actions_xt.empty:
    df_actions_xt["stock_code"] = df_actions_xt["stock_code"].map(normalize_stock_code)
    df_actions_xt["ex_date"] = pd.to_datetime(df_actions_xt["ex_date"], errors="coerce").dt.strftime("%Y-%m-%d")
df_actions_xt["import_time"] = import_time

left_keys = df_actions_gogoal[["stock_code", "ex_date"]].drop_duplicates()
right_keys = df_actions_xt[["stock_code", "ex_date"]].drop_duplicates()
df_actions_check = left_keys.merge(right_keys, on=["stock_code", "ex_date"], how="outer", indicator=True)
df_actions_check["is_consistent"] = df_actions_check["_merge"].eq("both")
df_actions_check["import_time"] = import_time
if df_actions_gogoal.empty and df_actions_xt.empty:
    df_actions_check = pd.DataFrame([{
        "stock_code": None,
        "ex_date": None,
        "_merge": "both",
        "is_consistent": True,
        "note": "Go-Goal 与迅投在查询窗口内均无公司行为事件",
        "import_time": import_time,
    }])

df_actions_merged = df_actions_gogoal.merge(
    df_actions_xt, on=["stock_code", "ex_date"], how="outer", suffixes=("_gogoal", "_xt")
)
df_actions_merged["import_time"] = import_time

daily_closes = fetch_stock_closes_range(
    gogoal_query,
    stock_codes=stock_codes,
    start_date=source_weight_date,
    end_date=target_date,
)

print("Daily close rows:", len(daily_closes))
print("Trading dates:", daily_closes["trade_date"].nunique())
print("Corporate action rows:", len(df_actions_merged))
display(daily_closes.head())
display(df_actions_merged.head())

正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库
正在尝试first_config连接: 192.168.1.30:3306
成功使用first_config连接数据库


Daily close rows: 3300
Trading dates: 11
Corporate action rows: 40


,trade_date,stock_code,stock_name,close_price
0,20260615,601319.SH,中国人保,7.16
1,20260615,688008.SH,澜起科技,240.10
2,20260615,601728.SH,中国电信,6.05
3,20260615,002074.SZ,国轩高科,29.35
4,20260615,300803.SZ,指南针,93.20


,stock_code,stock_name,declare_date,ex_date,beftax_maxcashdiv,beftax_mincashdiv,aftax_cashdiv,stockdiv_ratio,trans_ratio,bonus_ratio,...,time,interest,stockBonus,stockGift,allotNum,allotPrice,gugai,dr,import_time_xt,import_time
0,000333.SZ,美的集团,2026-06-23,2026-06-29,38.000000,None,38.000000,None,NaN,None,...,1.782662e+12,3.800000,0.0,0.0,0.0,0.0,0.0,1.050836,20260703-1315,20260703-1315
1,000617.SZ,中油资本,2026-06-17,2026-06-24,0.470000,None,0.470000,None,NaN,None,...,1.782230e+12,0.047000,0.0,0.0,0.0,0.0,0.0,1.006702,20260703-1315,20260703-1315
2,000725.SZ,京东方A,2026-06-12,2026-06-18,0.560000,None,0.560000,None,NaN,None,...,1.781712e+12,0.056000,0.0,0.0,0.0,0.0,0.0,1.009933,20260703-1315,20260703-1315
3,000768.SZ,中航西飞,2026-06-12,2026-06-23,1.500000,None,1.500000,None,NaN,None,...,1.782144e+12,0.150000,0.0,0.0,0.0,0.0,0.0,1.007338,20260703-1315,20260703-1315
4,002049.SZ,紫光国微,2026-06-23,2026-06-30,3.099995,None,3.099995,None,NaN,None,...,1.782749e+12,0.309999,0.0,0.0,0.0,0.0,0.0,1.003645,20260703-1315,20260703-1315


In [5]:
# stock_codes = source_weights["stock_code"].tolist()

# daily_closes = fetch_stock_closes_range(
#     gogoal_query,
#     stock_codes=stock_codes,
#     start_date=source_weight_date,
#     end_date=target_date,
# )
# corporate_actions = fetch_corporate_actions(
#     gogoal_query,
#     stock_codes=stock_codes,
#     start_date=source_weight_date,
#     end_date=target_date,
# )

# print("Daily close rows:", len(daily_closes))
# print("Trading dates:", daily_closes["trade_date"].nunique())
# print("Corporate action rows:", len(corporate_actions))
# display(daily_closes.head())
# display(corporate_actions.head())


## 推算并保存权重清单

In [6]:
corporate_actions = df_actions_merged
target_weights, trace, daily_summary, standardized_actions = project_weights_by_close_and_actions(
    source_weights=source_weights,
    daily_closes=daily_closes,
    corporate_actions=corporate_actions,
    target_date=target_date,
)

if output_dir is None:
    output_dir = make_projection_output_dir(PROJECT_ROOT)

projection_outputs = save_projection_outputs(
    target_weights=target_weights,
    trace=trace,
    daily_summary=daily_summary,
    source_weights=source_weights,
    standardized_actions=standardized_actions,
    output_dir=output_dir,
    target_date=target_date,
    index_code=INDEX_CODE,
)

print("Output dir:", projection_outputs.output_dir)
print("Target weights:", projection_outputs.target_path)
print("Daily summary:", projection_outputs.summary_path)
print("Saved projected weight files:", len(trace))
print("Projected weight sum pct:", target_weights["projected_weight_pct"].sum())
display(daily_summary.tail())
display(target_weights.head(20))


Output dir: E:\Codex\系统\Stock-Index-Fitting\data\weights_projection
Target weights: E:\Codex\系统\Stock-Index-Fitting\data\weights_projection\000300-20260630.csv
Daily summary: E:\Codex\系统\Stock-Index-Fitting\data\weights_projection\000300-20260630-daily_summary.csv
Saved projected weight files: 10
Projected weight sum pct: 100.0


,trade_date,stock_count,weight_sum_pct,action_count,share_action_count,cash_dividend_action_count
6,20260624,300,100.0,3,0,3
7,20260625,300,100.0,3,0,3
8,20260626,300,100.0,9,0,9
9,20260629,300,100.0,3,0,3
10,20260630,300,100.0,6,0,6


,trade_date,stock_code,stock_name,source_weight_date,source_weight_pct,projected_weight_pct,base_close,close_price,share_factor,relative_value,cumulative_cash_dividend_per_base_share
0,20260630,300308.SZ,中际旭创,20260615,4.998,5.006529,1245.00,1270.00,1.0,0.050984,0.00000
1,20260630,300750.SZ,宁德时代,20260615,3.790,3.674149,398.10,393.01,1.0,0.037415,0.00000
2,20260630,300502.SZ,新易盛,20260615,2.713,2.991968,540.49,607.00,1.0,0.030468,0.00000
3,20260630,600519.SH,贵州茅台,20260615,2.860,2.619331,1271.10,1185.49,1.0,0.026674,28.02423
4,20260630,603986.SH,兆易创新,20260615,1.245,1.923549,518.00,815.00,1.0,0.019588,0.00000
5,20260630,601318.SH,中国平安,20260615,2.081,1.798958,54.23,47.74,1.0,0.018320,0.00000
6,20260630,688256.SH,寒武纪,20260615,1.510,1.772198,1335.00,1595.55,1.0,0.018047,0.00000
7,20260630,600036.SH,招商银行,20260615,1.736,1.553734,38.95,35.50,1.0,0.015822,0.00000
8,20260630,601899.SH,紫金矿业,20260615,1.858,1.464988,31.31,25.14,1.0,0.014919,0.38000
9,20260630,002371.SZ,北方华创,20260615,1.063,1.353450,682.22,884.56,1.0,0.013783,0.00000


## 可选：查看某一天的推算轨迹

In [7]:
# 已默认保存源权重日之后、target_date 之前及当日的全部交易日推算结果。
# 每个文件表示：使用 current_date 闭市后的 close price 估计次日指数将使用的权重清单。
if trace:
    saved_dates = sorted(trace)
    print("Saved date range:", saved_dates[0], "->", saved_dates[-1])
    print("Saved files:", len(saved_dates))
    display(trace[saved_dates[-1]].head(20))
else:
    print("没有中间交易日需要保存。")


Saved date range: 20260616 -> 20260630
Saved files: 10


,trade_date,stock_code,stock_name,source_weight_date,source_weight_pct,projected_weight_pct,base_close,close_price,share_factor,relative_value,cumulative_cash_dividend_per_base_share
0,20260630,300308.SZ,中际旭创,20260615,4.998,5.006529,1245.00,1270.00,1.0,0.050984,0.00000
1,20260630,300750.SZ,宁德时代,20260615,3.790,3.674149,398.10,393.01,1.0,0.037415,0.00000
2,20260630,300502.SZ,新易盛,20260615,2.713,2.991968,540.49,607.00,1.0,0.030468,0.00000
3,20260630,600519.SH,贵州茅台,20260615,2.860,2.619331,1271.10,1185.49,1.0,0.026674,28.02423
4,20260630,603986.SH,兆易创新,20260615,1.245,1.923549,518.00,815.00,1.0,0.019588,0.00000
5,20260630,601318.SH,中国平安,20260615,2.081,1.798958,54.23,47.74,1.0,0.018320,0.00000
6,20260630,688256.SH,寒武纪,20260615,1.510,1.772198,1335.00,1595.55,1.0,0.018047,0.00000
7,20260630,600036.SH,招商银行,20260615,1.736,1.553734,38.95,35.50,1.0,0.015822,0.00000
8,20260630,601899.SH,紫金矿业,20260615,1.858,1.464988,31.31,25.14,1.0,0.014919,0.38000
9,20260630,002371.SZ,北方华创,20260615,1.063,1.353450,682.22,884.56,1.0,0.013783,0.00000
